In [1]:
pip install langchain pypdf sentence-transformers chromadb langchain_community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/1

In [24]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Medical_book.pdf")
documents = loader.load()

print(len(documents))

637


In [25]:
def clean_text(text):
    text = text.replace("\n", " ")
    text = text.replace("GALE ENCYCLOPEDIA OF MEDICINE", "")
    return text

for doc in documents:
    doc.page_content = clean_text(doc.page_content)

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

4196


In [27]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- CHUNK {i} ---\n")
    print(chunk.page_content)


--- CHUNK 0 ---

The GALE ENCYCLOPEDIA of MEDICINE SECOND EDITION

--- CHUNK 1 ---

The GALE ENCYCLOPEDIA of MEDICINE SECOND EDITION JACQUELINE L. LONGE, EDITOR DEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR VOLUME A-B 1

--- CHUNK 2 ---

STAFF Jacqueline L. Longe, Project Editor Deirdre S. Blanchfield, Associate Editor Christine B. Jeryan, Managing Editor Donna Olendorf, Senior Editor Stacey Blachford, Associate Editor Kate Kretschmann, Melissa C. McDade, Ryan Thomason, Assistant Editors Mark Springer, Technical Specialist Andrea Lopeman, Programmer/Analyst Barbara J. Yarrow,Manager, Imaging and Multimedia Content Robyn V . Young,Project Manager, Imaging and Multimedia Content Dean Dauphinais, Senior Editor, Imaging and Multimedia Content Kelly A. Quin, Editor, Imaging and Multimedia Content Leitha Etheridge-Sims, Mary K. Grimes, Dave Oblender, Image Catalogers Pamela A. Reed, Imaging Coordinator Randy Bassett, Imaging Supervisor Robert Duncan, Senior Imaging Specialist Dan Newell, Imaging

In [28]:
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
    chunk.metadata["source"] = "Gale Encyclopedia of Medicine"
    chunk.metadata["page"] = chunk.metadata.get("page", "unknown")

In [29]:
lengths = [len(c.page_content) for c in chunks]

print("Average length:", sum(lengths)//len(lengths))
print("Max length:", max(lengths))
print("Min length:", min(lengths))

Average length: 745
Max length: 800
Min length: 48


In [30]:
unique_texts = set()
unique_chunks = []

for c in chunks:
    if c.page_content not in unique_texts:
        unique_chunks.append(c)
        unique_texts.add(c.page_content)

chunks = unique_chunks

print("Chunks after cleaning:", len(chunks))

Chunks after cleaning: 4196


In [31]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
from langchain_community.vectorstores import Chroma

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./medical_db"
)

In [33]:
def classify_query(q):
    q = q.lower()

    if "treatment" in q:
        return "Treatment"
    elif "symptom" in q:
        return "Symptoms"
    elif "diagnosis" in q:
        return "Diagnosis"
    elif "what is" in q or "definition" in q:
        return "Definition"
    else:
        return "General"

In [38]:
def search_medical(query, k=10):

    print("\nQuery Type:", classify_query(query))

    results = vectordb.similarity_search_with_score(query, k=k)

    for i, (doc, score) in enumerate(results):

        confidence = 1/(1+score)

        print("\n==============================")
        print(f"Result {i+1}")
        print("Page:", doc.metadata["page"])
        print("Confidence:", round(confidence*100,2), "%")
        print("\nText:\n")
        print(doc.page_content[:600])

In [39]:
search_medical("What is abdominal ultrasound?")


Query Type: Definition

Result 1
Page: 17
Confidence: 70.11 %

Text:

ultrasound remains faster and less expensive than computed tomography scans (CT), its primary rival in abdominal imaging. Furthermore, as abdominal ultrasounds are generally undertaken as “medically necessary” procedures designed to detect the presence of suspected abnormalities, they are covered  24 Abdominal ultrasound GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 4

Result 2
Page: 17
Confidence: 70.11 %

Text:

ultrasound remains faster and less expensive than computed tomography scans (CT), its primary rival in abdominal imaging. Furthermore, as abdominal ultrasounds are generally undertaken as “medically necessary” procedures designed to detect the presence of suspected abnormalities, they are covered  24 Abdominal ultrasound GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 4

Result 3
Page: 17
Confidence: 70.11 %

Text:

ultrasound remains faster and less expensive than computed tomography scans (CT), its prima

In [36]:
retriever = vectordb.as_retriever(search_kwargs={"k":3})

docs = retriever.invoke(
    "What is abdominal ultrasound?"
)

print(docs[0].page_content)

ultrasound remains faster and less expensive than computed tomography scans (CT), its primary rival in abdominal imaging. Furthermore, as abdominal ultrasounds are generally undertaken as “medically necessary” procedures designed to detect the presence of suspected abnormalities, they are covered  24 Abdominal ultrasound GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 4


In [37]:
retriever = vectordb.as_retriever(search_kwargs={"k":3})

docs = retriever.invoke(
    "cure of cancer?"
)

print(docs[0].page_content)

loss, loss of appetite, diarrhea, mouth sores, fatigue, shortness of breath, and a weakened immune system. Alternative treatment Research suggests acupuncture can help manage chemotherapy-related nausea and vomiting and control pain associated with surgery. Prognosis Anal cancer is often curable. The chance of recovery depends on the cancer stage and the patient’s general health. KEY TERMS Biopsy—A procedure in which a small piece of body tissue is removed and examined under a microscope for cancer. Chemotherapy —A cancer treatment in which drugs delivered into the blood stream kill cancer cells or make them more vulnerable to radiation therapy. Human papillomavirus (HPV)—A virus with many subtypes, some of which cause cell changes that increase the risk of certain cancers. Human


In [15]:
from langchain_core.prompts import PromptTemplate

template = """
You are a medical assistant.

Answer ONLY using the provided context.
If answer is not in context, say:
"I could not find this in the medical encyclopedia."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)